In [3]:
from datasets import load_dataset

# ==========================
#    DATASET FOR TESTING
# ==========================

SAMPLE_SIZE = 500000
OUTPUT_SAMPLE_PATH = "WLOSKI_DATASET_SAMPLE.txt"

print(f"Downloading sample ({SAMPLE_SIZE} articles)...")

dataset_sample = load_dataset(
    "wikimedia/wikipedia", "20231101.it", split=f"train[:{SAMPLE_SIZE}]"
)

with open(OUTPUT_SAMPLE_PATH, "w", encoding="utf-8") as f:
  for item in dataset_sample:
    f.write(item["text"] + "\n\n")

print(f"Done! Saved {len(dataset_sample)} articles to {OUTPUT_SAMPLE_PATH}")
print(f"Preview of the first article:\n{dataset_sample[0]['text'][:300]}...")

Done! Saved 500000 articles to WLOSKI_DATASET_SAMPLE.txt
Preview of the first article:
Larmonium, o harmonium, è un tipo di organo a serbatoio d'aria con una (o più) tastiere e due pedali per azionare i mantici per l'aria. Il suono è prodotto da una o più file di ance libere, analogamente alla fisarmonica e all'armonica a bocca. 

Nella classificazione Hornbostel-Sachs, rientra nella ...


In [ ]:
from datasets import load_dataset

# ==========================
#   FULL DATASET (~5GB)
# ==========================

print(f"Downloading full dataset!")

dataset_full = load_dataset("wikimedia/wikipedia", "20231101.it", split="train")

print(
    f"Installed, amount of articles:"
    f" {len(dataset_full):,}"
)

In [4]:
# ==========================================
# CZYSZCZENIE TEKSTU Z NIEWŁOSKICH SŁÓW
# ==========================================
import os
import re
import urllib.request

INPUT_SAMPLE_PATH = "WLOSKI_DATASET_SAMPLE.txt"
OUTPUT_CLEANED_CORPUS = "WLOSKI_KORPUS_OCZYSZCZONY.txt"
DICT_FILE = "it_IT.dic"

# 1. Pobranie i wczytanie słownika języka włoskiego
if not os.path.exists(DICT_FILE):
    print("Pobieranie oficjalnego słownika języka włoskiego...")
    url = "https://raw.githubusercontent.com/LibreOffice/dictionaries/master/it_IT/it_IT.dic"
    urllib.request.urlretrieve(url, DICT_FILE)

print("Wczytywanie słownika do pamięci RAM...")
italian_lexicon = set()
with open(DICT_FILE, "r", encoding="latin-1") as f:
    for line in f:
        word = line.strip().split("/")[0].lower()
        if word:
            italian_lexicon.add(word)

# Włoskie krótkie słowa (spójniki, przyimki, formy po apostrofie)
ALLOWED_SHORT_WORDS = {"a", "e", "i", "o", "u", "d", "l", "c", "m", "t", "s", "v", "n"}
known_valid = set(ALLOWED_SHORT_WORDS)
known_invalid = set()

HYPHENS_AND_APOSTROPHES = re.compile(r"[-–—'’]")
NON_LETTER_CLEANER = re.compile(r"[^a-zàèéìòù\s]")
ITALIAN_WORD_REGEX = re.compile(r"^[a-zàèéìòù]+$")

print(f"\nGenerowanie czystego korpusu: {OUTPUT_CLEANED_CORPUS}...")

line_count = 0
with open(INPUT_SAMPLE_PATH, "r", encoding="utf-8") as f_in, \
        open(OUTPUT_CLEANED_CORPUS, "w", encoding="utf-8") as f_out:
    for line in f_in:
        line_count += 1
        if line_count % 500000 == 0:
            print(f"Przetworzono {line_count:,} linii...")

        text = line.lower()
        text = HYPHENS_AND_APOSTROPHES.sub(" ", text)
        text = NON_LETTER_CLEANER.sub(" ", text)

        valid_words_in_line = []

        for token in text.split():
            is_valid = False

            if token in known_valid:
                is_valid = True
            elif token in known_invalid or len(token) == 1 or not ITALIAN_WORD_REGEX.match(token):
                known_invalid.add(token)
            elif token in italian_lexicon:
                known_valid.add(token)
                is_valid = True
            else:
                known_invalid.add(token)

            if is_valid:
                valid_words_in_line.append(token)

        # Zapisujemy oczyszczoną linijkę
        if valid_words_in_line:
            f_out.write(" ".join(valid_words_in_line) + "\n")

print(f"Zapisano oczyszczony tekst do {OUTPUT_CLEANED_CORPUS}.")

Wczytywanie słownika do pamięci RAM...

Generowanie czystego korpusu: WLOSKI_KORPUS_OCZYSZCZONY.txt...
Przetworzono 500,000 linii...
Przetworzono 1,000,000 linii...
Przetworzono 1,500,000 linii...
Przetworzono 2,000,000 linii...
Przetworzono 2,500,000 linii...
Przetworzono 3,000,000 linii...
Przetworzono 3,500,000 linii...
Przetworzono 4,000,000 linii...
Przetworzono 4,500,000 linii...
Przetworzono 5,000,000 linii...
Przetworzono 5,500,000 linii...
Przetworzono 6,000,000 linii...
Przetworzono 6,500,000 linii...
Przetworzono 7,000,000 linii...
Przetworzono 7,500,000 linii...
Przetworzono 8,000,000 linii...
Przetworzono 8,500,000 linii...
Przetworzono 9,000,000 linii...
Przetworzono 9,500,000 linii...
Przetworzono 10,000,000 linii...
Przetworzono 10,500,000 linii...
Przetworzono 11,000,000 linii...
Przetworzono 11,500,000 linii...
Przetworzono 12,000,000 linii...
Przetworzono 12,500,000 linii...
Przetworzono 13,000,000 linii...
Przetworzono 13,500,000 linii...
Przetworzono 14,000,000 lin

In [5]:
# ==========================================
# ZLICZANIE CZĘSTOTLIWOŚCI LITER
# ==========================================
from collections import Counter

OUTPUT_CLEANED_CORPUS = "WLOSKI_KORPUS_OCZYSZCZONY.txt"

print(f"Zliczanie liter z pliku {OUTPUT_CLEANED_CORPUS}...")
char_counter = Counter()

line_count = 0
with open(OUTPUT_CLEANED_CORPUS, "r", encoding="utf-8") as f_in:
    for line in f_in:
        line_count += 1
        if line_count % 500000 == 0:
            print(f"Przetworzono {line_count:,} linii...")

        # Usuwamy białe znaki (spacje, nowa linia), żeby liczyć TYLKO litery
        chars_only = line.replace(" ", "").replace("\n", "")
        char_counter.update(chars_only)

print(f"Zliczanie zakończone. Znaleziono {len(char_counter)} unikalnych włoskich liter.")

# Wyświetlenie podglądu 5 najpopularniejszych liter
print("\nTop 5 najpopularniejszych liter:")
for char, freq in char_counter.most_common(5):
    print(f"'{char}': {freq:,}")

Zliczanie liter z pliku WLOSKI_KORPUS_OCZYSZCZONY.txt...
Przetworzono 500,000 linii...
Przetworzono 1,000,000 linii...
Przetworzono 1,500,000 linii...
Przetworzono 2,000,000 linii...
Przetworzono 2,500,000 linii...
Przetworzono 3,000,000 linii...
Przetworzono 3,500,000 linii...
Przetworzono 4,000,000 linii...
Przetworzono 4,500,000 linii...
Przetworzono 5,000,000 linii...
Przetworzono 5,500,000 linii...
Przetworzono 6,000,000 linii...
Przetworzono 6,500,000 linii...
Przetworzono 7,000,000 linii...
Przetworzono 7,500,000 linii...
Przetworzono 8,000,000 linii...
Przetworzono 8,500,000 linii...
Przetworzono 9,000,000 linii...
Przetworzono 9,500,000 linii...
Przetworzono 10,000,000 linii...
Przetworzono 10,500,000 linii...
Przetworzono 11,000,000 linii...
Zliczanie zakończone. Znaleziono 26 unikalnych włoskich liter.

Top 5 najpopularniejszych liter:
'e': 92,042,151
'a': 82,213,990
'o': 75,425,185
'i': 72,420,510
'n': 58,053,684


In [6]:
# ==========================================
# KOMÓRKA 3: BUDOWA DRZEWA I GENEROWANIE KODÓW HUFFMANA
# ==========================================
import heapq

OUTPUT_HUFFMAN_PATH = "WLOSKI_HUFFMAN.txt"


class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq


def generate_huffman_codes(node, current_code="", huffman_dict=None):
    if huffman_dict is None:
        huffman_dict = {}
    if node is not None:
        if node.char is not None:
            huffman_dict[node.char] = current_code
        generate_huffman_codes(node.left, current_code + "0", huffman_dict)
        generate_huffman_codes(node.right, current_code + "1", huffman_dict)
    return huffman_dict


print("Budowanie drzewa Huffmana...")

# Zamiana wyników z char_counter na węzły początkowe
priority_queue = [HuffmanNode(char, freq) for char, freq in char_counter.items()]
heapq.heapify(priority_queue)

# Algorytm łączenia węzłów
while len(priority_queue) > 1:
    left = heapq.heappop(priority_queue)
    right = heapq.heappop(priority_queue)

    merged = HuffmanNode(None, left.freq + right.freq)
    merged.left = left
    merged.right = right
    heapq.heappush(priority_queue, merged)

# Generowanie kodów z korzenia
huffman_codes = generate_huffman_codes(priority_queue[0])

# Zapis i wyświetlanie wyników
print("\n=== WYNIKI (LITERA : CZĘSTOTLIWOŚĆ -> KOD HUFFMANA) ===")
with open(OUTPUT_HUFFMAN_PATH, "w", encoding="utf-8") as f_out:
    f_out.write("LITERA\tCZESTOTLIWOSC\tKOD_HUFFMANA\n")
    for char, freq in char_counter.most_common():
        code = huffman_codes[char]
        line = f"'{char}'\t{freq}\t{code}"
        print(line)
        f_out.write(line + "\n")

print(f"\nGotowe! Zapisano kody Huffmana do pliku: {OUTPUT_HUFFMAN_PATH}")

Budowanie drzewa Huffmana...

=== WYNIKI (LITERA : CZĘSTOTLIWOŚĆ -> KOD HUFFMANA) ===
'e'	92042151	100
'a'	82213990	011
'o'	75425185	001
'i'	72420510	000
'n'	58053684	1101
'l'	54294970	1100
'r'	47267863	1011
't'	43301749	0101
's'	33359537	11111
'c'	31305618	11101
'd'	28342188	11100
'u'	22150246	10100
'm'	18666297	01001
'p'	17975432	01000
'g'	14636422	101011
'f'	8527914	1111011
'v'	8411263	1111010
'z'	8315288	1111001
'b'	7476461	1111000
'h'	5744401	1010101
'q'	2420918	10101001
'k'	488039	1010100011
'y'	357322	1010100001
'w'	352995	1010100000
'x'	265394	10101000101
'j'	108178	10101000100

Gotowe! Zapisano kody Huffmana do pliku: WLOSKI_HUFFMAN.txt
